## Part 1: Model Training & Implementation

### Dataset Selection: SQuAD v1.1 for Question Answering

**Why SQuAD v1.1? A Balanced Technical Approach:**

**Task Complexity**: Question Answering provides the perfect balance to evaluate all three architectures:
- **GPT-2 (Decoder-only)**: Can generate answers but lacks bidirectional context understanding
- **BERT (Encoder-only)**: Excels at extractive QA by finding answer spans in context
- **T5 (Encoder-decoder)**: Naturally handles generative QA with "question: [Q] context: [C]" format

**Architectural Fairness**: Unlike pure generation tasks (favoring GPT-2) or classification tasks (favoring BERT), QA requires both:
- **Understanding**: Comprehending question and context relationships
- **Generation/Extraction**: Producing coherent answers
 **Evaluation Balance**:
- **F1 Score**: Measures token-level overlap (good for all models)
- **Exact Match**: Tests precise answer extraction
- **BLEU**: Evaluates answer quality for generative approaches
 **Technical Implementation**: Each architecture handles QA differently:
- **GPT-2**: Generative QA with prompt: "Context: [C] Question: [Q] Answer:"
- **BERT**: Extractive QA predicting start/end positions in context
- **T5**: Text-to-text QA with format: "question: [Q] context: [C]"

**Dataset Size**: Using 500 samples for fast, educational training while maintaining meaningful comparisons.

In [1]:
# Import required libraries
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM, TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, DataCollatorForSeq2Seq
)
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Check device (should show cuda:0 in Colab with GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU - training will be slower")

Using device: cuda
GPU Memory: 15.8 GB
GPU Name: Tesla T4


### Dataset Preparation and Preprocessing

Now let's load and explore SQuAD v1.1 to understand its structure:

In [2]:
# Load SQuAD v1.1 dataset
print("Loading SQuAD v1.1 dataset...")
dataset = load_dataset("squad")

print("Dataset structure:")
print(dataset)

# Show dataset splits
print(f"\nDataset splits:")
print(f"Train: {len(dataset['train'])} examples")
print(f"Validation: {len(dataset['validation'])} examples")

# Explore sample structure
sample = dataset["train"][0]
print(f"\n📖 Sample QA Example:")
print(f"ID: {sample['id']}")
print(f"Context: {sample['context'][:200]}...")
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']['text'][0]}")
print(f"Answer Start: {sample['answers']['answer_start'][0]}")


print(f"\n Using subset for educational purposes:")
train_dataset = dataset["train"].shuffle(seed=42).select(range(500))
val_dataset = dataset["validation"].shuffle(seed=42).select(range(100))

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


contexts_lengths = [len(sample['context'].split()) for sample in train_dataset]
questions_lengths = [len(sample['question'].split()) for sample in train_dataset]
answers_lengths = [len(sample['answers']['text'][0].split()) for sample in train_dataset]

print(f"\n Dataset Statistics (500 samples):")
print(f"Avg context length: {np.mean(contexts_lengths):.1f} words")
print(f"Avg question length: {np.mean(questions_lengths):.1f} words")
print(f"Avg answer length: {np.mean(answers_lengths):.1f} words")

Loading SQuAD v1.1 dataset...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Dataset splits:
Train: 87599 examples
Validation: 10570 examples

📖 Sample QA Example:
ID: 5733be284776f41900661182
Context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta...
Question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer: Saint Bernadette Soubirous
Answer Start: 515

 Using subset for educational purposes:
Training samples: 500
Validation samples: 100

 Dataset Statistics (500 samples):
Avg context length: 118.2 words
Avg question length: 9.9 words
Avg answer length: 3.2 words


## Model Implementation

### 1. GPT-2 (Decoder-only) - Generative Question Answering

GPT-2 will generate answers by continuing the prompt: "Context: [C] Question: [Q] Answer:"

In [3]:
# GPT-2 setup for generative QA
print("Setting up GPT-2 for Generative Question Answering...")

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2")

# Move model to GPU
gpt2_model = gpt2_model.to(device)
print(f"GPT-2 moved to: {device}")

# GPT-2 preprocessing for QA
def preprocess_gpt2_qa(examples):
    """Convert SQuAD to GPT-2 format: 'Context: [C] Question: [Q] Answer: [A]'"""
    inputs = []

    for i in range(len(examples['context'])):
        context = examples['context'][i]
        question = examples['question'][i]
        answer = examples['answers'][i]['text'][0]  # Take first answer

        # Create prompt for GPT-2
        prompt = f"Context: {context[:200]} Question: {question} Answer: {answer}"
        inputs.append(prompt)

    # Tokenize
    tokenized = gpt2_tokenizer(
        inputs,
        max_length=384,  # Shorter for GPT-2 memory efficiency
        truncation=True,
        padding="max_length"
    )

    return tokenized

# Preprocess datasets
print("Preprocessing SQuAD for GPT-2...")
train_gpt2 = train_dataset.map(preprocess_gpt2_qa, batched=True, remove_columns=train_dataset.column_names)
val_gpt2 = val_dataset.map(preprocess_gpt2_qa, batched=True, remove_columns=val_dataset.column_names)

print(f"GPT-2 training samples: {len(train_gpt2)}")

# Training setup
gpt2_training_args = TrainingArguments(
    output_dir="./gpt2-squad",
    num_train_epochs=1,  # Quick training for demo
    per_device_train_batch_size=2,  # Small batch for memory efficiency
    per_device_eval_batch_size=2,
    save_steps=1000,
    logging_steps=50,
    report_to=[],
    learning_rate=5e-5,
    warmup_steps=50,
)

# Data collator for causal LM
gpt2_collator = DataCollatorForLanguageModeling(
    tokenizer=gpt2_tokenizer,
    mlm=False  # Causal LM
)

gpt2_trainer = Trainer(
    model=gpt2_model,
    args=gpt2_training_args,
    train_dataset=train_gpt2,
    eval_dataset=val_gpt2,
    data_collator=gpt2_collator,
)

print("Training GPT-2 on SQuAD... (2-3 minutes)")
print("Progress format: step/total_steps")
gpt2_trainer.train()

Setting up GPT-2 for Generative Question Answering...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

GPT-2 training samples: 500
Training GPT-2 on SQuAD... (2-3 minutes)
Progress format: step/total_steps


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.778900
100,3.587100
150,3.503200
200,3.568800
250,3.432700


TrainOutput(global_step=250, training_loss=3.5741444091796875, metrics={'train_runtime': 61.84, 'train_samples_per_second': 8.085, 'train_steps_per_second': 4.043, 'total_flos': 97984512000000.0, 'train_loss': 3.5741444091796875, 'epoch': 1.0})

In [17]:
sample = "Context: The Nile is the longest river in the world, flowing through eleven countries in Africa. Question: What is the longest river in the world? Answer:"
inputs = gpt2_tokenizer(sample, return_tensors="pt").to(device)
outputs = gpt2_model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=50,
    do_sample=True,
    temperature=0.7,
    pad_token_id=gpt2_tokenizer.eos_token_id
)
generated = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = generated.split("Answer:")[-1].split("Question:")[0].strip()
print("Extracted Answer:", answer)


Extracted Answer: the Nile. The river is the longest and most popular river in Africa.


### 2. BERT (Encoder-only) - Extractive Question Answering

BERT will find answer spans in the context by predicting start and end positions.

In [5]:
# BERT setup for extractive QA
print("Setting up BERT for Extractive Question Answering...")

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

# Move model to GPU
bert_model = bert_model.to(device)
print(f"BERT moved to: {device}")

# BERT preprocessing for QA
def preprocess_bert_qa(examples):
    """Prepare SQuAD for BERT extractive QA"""
    questions = examples['question']
    contexts = examples['context']

    # Tokenize question-context pairs
    tokenized_examples = bert_tokenizer(
        questions,
        contexts,
        truncation="only_second",  # Truncate context if too long
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Find start and end positions for answers
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(bert_tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Find token start and end positions
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # Check if answer is in this chunk
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # Move to start and end of answer
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

# Preprocess datasets
print("Preprocessing SQuAD for BERT...")
train_bert = train_dataset.map(preprocess_bert_qa, batched=True, remove_columns=train_dataset.column_names)
val_bert = val_dataset.map(preprocess_bert_qa, batched=True, remove_columns=val_dataset.column_names)

print(f"BERT training samples: {len(train_bert)}")

# Training setup
bert_training_args = TrainingArguments(
    output_dir="./bert-squad",
    num_train_epochs=1,
    per_device_train_batch_size=4,  # BERT can handle larger batches
    per_device_eval_batch_size=4,
    save_steps=1000,
    logging_steps=50,
    report_to=[],
    learning_rate=3e-5,
    warmup_steps=50,
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_training_args,
    train_dataset=train_bert,
    eval_dataset=val_bert,
    tokenizer=bert_tokenizer,
)

print("Training BERT on SQuAD... (2-3 minutes)")
bert_trainer.train()

Setting up BERT for Extractive Question Answering...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

BERT training samples: 503
Training BERT on SQuAD... (2-3 minutes)


Step,Training Loss
50,5.503800
100,4.283600


TrainOutput(global_step=126, training_loss=4.708474477132161, metrics={'train_runtime': 42.7312, 'train_samples_per_second': 11.771, 'train_steps_per_second': 2.949, 'total_flos': 98574201478656.0, 'train_loss': 4.708474477132161, 'epoch': 1.0})

In [18]:

context = "The Nile is the longest river in the world, flowing through eleven countries in Africa."
question = "What is the longest river in the world?"

inputs = bert_tokenizer(question, context, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = bert_model(**inputs)
start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1

answer = bert_tokenizer.convert_tokens_to_string(
    bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][start_idx:end_idx])
)

print("Extracted Answer:", answer)


Extracted Answer: nile


### 3. T5 (Encoder-Decoder) - Text-to-Text Question Answering

T5 will handle QA as text-to-text generation with format: "question: [Q] context: [C]" → answer

In [7]:
# T5 setup for text-to-text QA
print("Setting up T5 for Text-to-Text Question Answering...")

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

# Move model to GPU
t5_model = t5_model.to(device)
print(f"T5 moved to: {device}")

# T5 preprocessing for QA
def preprocess_t5_qa(examples):
    """Convert SQuAD to T5 format: 'question: [Q] context: [C]' → answer"""
    inputs = []
    targets = []

    for i in range(len(examples['context'])):
        context = examples['context'][i][:300]  # Truncate context for memory
        question = examples['question'][i]
        answer = examples['answers'][i]['text'][0]  # Take first answer

        # T5 format: "question: [Q] context: [C]"
        input_text = f"question: {question} context: {context}"
        inputs.append(input_text)
        targets.append(answer)

    # Tokenize inputs
    model_inputs = t5_tokenizer(
        inputs,
        max_length=384,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets
    labels = t5_tokenizer(
        targets,
        max_length=64,  # Answers are usually short
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Preprocess datasets
print("Preprocessing SQuAD for T5...")
train_t5 = train_dataset.map(preprocess_t5_qa, batched=True, remove_columns=train_dataset.column_names)
val_t5 = val_dataset.map(preprocess_t5_qa, batched=True, remove_columns=val_dataset.column_names)

print(f"T5 training samples: {len(train_t5)}")

# Training setup
t5_training_args = TrainingArguments(
    output_dir="./t5-squad",
    num_train_epochs=1,
    per_device_train_batch_size=2,  # T5 uses more memory
    per_device_eval_batch_size=2,
    save_steps=1000,
    logging_steps=50,
    report_to=[],
    learning_rate=3e-4,  # T5 often needs higher learning rate
    warmup_steps=50,
)

# Data collator for seq2seq
t5_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=t5_model
)

t5_trainer = Trainer(
    model=t5_model,
    args=t5_training_args,
    train_dataset=train_t5,
    eval_dataset=val_t5,
    data_collator=t5_collator,
    tokenizer=t5_tokenizer,
)

print("Training T5 on SQuAD... (2-3 minutes)")
t5_trainer.train()

Setting up T5 for Text-to-Text Question Answering...


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

T5 training samples: 500
Training T5 on SQuAD... (2-3 minutes)


Step,Training Loss
50,9.210400
100,0.447500
150,0.242600
200,0.276000
250,0.216800


TrainOutput(global_step=250, training_loss=2.0786471710205077, metrics={'train_runtime': 45.0922, 'train_samples_per_second': 11.088, 'train_steps_per_second': 5.544, 'total_flos': 50753175552000.0, 'train_loss': 2.0786471710205077, 'epoch': 1.0})

In [8]:
# T5 test with Nile example
context = "The Nile is the longest river in the world, flowing through eleven countries in Africa."
question = "What is the longest river in the world?"

sample = f"question: {question} context: {context}"
inputs = t5_tokenizer(sample, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = t5_model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=50
    )

answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("T5 Answer:", answer)


T5 Answer: The Nile


BERT (Extractive QA)

Output: nile

Pulled the exact span from context.

Always constrained to words inside the passage.

Plain, lowercase (because of bert-base-uncased).

GPT-2 (Generative QA)

Output: the Great Nile. Nile is one of the Great River Rivers in the world. According to

Found the right answer but rambled with extra made-up text.

Can hallucinate because it’s free-form generation.

Doesn’t know when to stop.

T5 (Text-to-Text QA)

Output: The Nile

Clean, short, well-formatted answer.

Generates like GPT-2, but trained in QA-friendly text-to-text style.

More concise than GPT-2, more natural than BERT.

## Training Documentation

### Training Setup Summary

| Model | Architecture | Task Format | Batch Size | Learning Rate | Max Length |
|-------|--------------|-------------|------------|---------------|------------|
| GPT-2 | Decoder-only | Generative QA | 2 | 5e-5 | 384 tokens |
| BERT | Encoder-only | Extractive QA | 4 | 3e-5 | 384 tokens |
| T5 | Encoder-decoder | Text-to-text QA | 2 | 3e-4 | 384/64 tokens |

**Common Settings:**
- **Optimizer**: AdamW (default in Transformers)
- **Epochs**: 1 (for quick demonstration)
- **Hardware**: Google Colab GPU (NVIDIA T4)
- **Dataset**: 500 training samples, 100 validation samples
- **Warmup Steps**: 50 for all models

### Why Different Approaches for Same Task?

**Task Adaptation Strategy**: Each architecture naturally handles QA differently:

1. **GPT-2 (Generative)**: Treats QA as text completion
   - Input: "Context: [C] Question: [Q] Answer:"
   - Output: Continues with the answer
   - Challenge: May hallucinate or ignore context

2. **BERT (Extractive)**: Finds exact answer spans in context
   - Input: [CLS] question [SEP] context [SEP]
   - Output: Start/end position predictions
   - Advantage: Always grounded in provided context

3. **T5 (Text-to-Text)**: Flexible generative approach
   - Input: "question: [Q] context: [C]"
   - Output: Generated answer text
   - Balance: Can extract or paraphrase from context

In [21]:
# Training Progress Summary and Loss Analysis
print("="*60)
print("TRAINING PROGRESS SUMMARY")
print("="*60)

# Collect training results
models_info = {
    "GPT-2": {
        "trainer": gpt2_trainer,
        "approach": "Generative QA",
        "batch_size": 2,
        "params": "124M",
        "memory_efficient": "Medium"  # decoder-only, moderate footprint
    },
    "BERT": {
        "trainer": bert_trainer,
        "approach": "Extractive QA",
        "batch_size": 4,
        "params": "110M",
        "memory_efficient": "Low"     # encoder-only, most efficient
    },
    "T5": {
        "trainer": t5_trainer,
        "approach": "Text-to-Text QA",
        "batch_size": 2,
        "params": "60M",
        "memory_efficient": "High"    # encoder+decoder, heavier on memory
    }
}

print(f"{'Model':<8} {'Approach':<15} {'Batch Size':<12} {'Parameters':<12} {'Memory Use'}")
print("-" * 70)

for model_name, info in models_info.items():
    print(f"{model_name:<8} {info['approach']:<15} {info['batch_size']:<12} {info['params']:<12} {info['memory_efficient']}")

# Extract final training losses
print(f"\nFINAL TRAINING LOSSES:")
for model_name, info in models_info.items():
    try:
        final_log = info['trainer'].state.log_history[-1]
        train_loss = final_log.get('train_loss', 'N/A')
        print(f"{model_name}: {train_loss:.4f}" if train_loss != 'N/A' else f"{model_name}: {train_loss}")
    except:
        print(f"{model_name}: Training log not available")

print(f"\n Training completed on {device}")
print(f"Total dataset: 500 training samples, 100 validation samples")


print("\nNote: Training losses are *not directly comparable across models* because each")
print("architecture uses a different objective (causal LM for GPT-2, span prediction")
print("for BERT, seq2seq generation for T5). Use evaluation metrics (EM, F1, BLEU)")
print("for fair cross-model comparison.\n")


TRAINING PROGRESS SUMMARY
Model    Approach        Batch Size   Parameters   Memory Use
----------------------------------------------------------------------
GPT-2    Generative QA   2            124M         Medium
BERT     Extractive QA   4            110M         Low
T5       Text-to-Text QA 2            60M          High

FINAL TRAINING LOSSES:
GPT-2: 3.5741
BERT: 4.7085
T5: 2.0786

 Training completed on cuda
Total dataset: 500 training samples, 100 validation samples

Note: Training losses are *not directly comparable across models* because each
architecture uses a different objective (causal LM for GPT-2, span prediction
for BERT, seq2seq generation for T5). Use evaluation metrics (EM, F1, BLEU)
for fair cross-model comparison.



### Difficulties Faced and Solutions

#### **Memory Management Challenges**
- **Problem**: GPU memory limitations with larger batch sizes
- **Solution**:
  - GPT-2: Reduced batch size to 2, truncated inputs to 384 tokens
  - BERT: Used batch size 4 (most memory efficient for QA)
  - T5: Smallest batch size (2) due to encoder-decoder architecture

####  **Preprocessing Complexity**
- **Problem**: Each architecture requires different input formats
- **Solution**: Created specialized preprocessing functions:
  - GPT-2: Simple concatenation with "Answer:" prompt
  - BERT: Complex start/end position calculation for extractive QA
  - T5: Clean "question: context:" format for seq2seq

####  **Model-Specific Challenges**

**GPT-2 Issues:**
- **Problem**: Attention mask warnings (pad_token = eos_token)
- **Solution**: Explicitly handled attention masks in generation

**BERT Issues:**
- **Problem**: Complex answer span calculation for SQuAD format
- **Solution**: Implemented robust tokenization with offset mapping

**T5 Issues:**
- **Problem**: Higher memory usage despite smaller parameter count
- **Solution**: Reduced context length to 300 characters, answer length to 64 tokens

#### **Training Convergence**
- **Problem**: Different convergence rates across architectures
- **Solution**: Adjusted learning rates:
  - GPT-2: 5e-5 (standard for causal LM)
  - BERT: 3e-5 (standard for QA fine-tuning)  
  - T5: 3e-4 (higher rate typical for T5)


# Part 2: Evaluation & Analysis

## Model Evaluation Setup

Now we'll evaluate all three trained models on the same QA task using multiple metrics and generate sample outputs for qualitative comparison.

In [10]:
import string
import re
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Install nltk if not available
try:
    import nltk
    nltk.download('punkt', quiet=True)
except:
    !pip install nltk
    import nltk
    nltk.download('punkt', quiet=True)

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    """Calculate F1 score between prediction and ground truth."""
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()

    if len(prediction_tokens) == 0 or len(ground_truth_tokens) == 0:
        return int(prediction_tokens == ground_truth_tokens)

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)

    return f1

def exact_match(prediction, ground_truth):
    """Check if the prediction exactly matches the ground truth after normalization."""
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def bleu_score(prediction, ground_truth):
    """Calculate BLEU score for prediction against ground truth."""
    if not prediction.strip():
        return 0.0

    reference = [ground_truth.split()]
    candidate = prediction.split()

    smoothie = SmoothingFunction().method4
    return sentence_bleu(reference, candidate, smoothing_function=smoothie)

print("Evaluation metrics defined successfully!")

Evaluation metrics defined successfully!


# Part 2: Evaluation & Analysis

## 5. Model Evaluation

Now we'll evaluate all three models on the same QA task using multiple metrics including F1 score, Exact Match (EM), and BLEU score. We'll also provide qualitative analysis with sample outputs.

In [11]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from collections import Counter
import re
import numpy as np

# Download required NLTK data
nltk.download('punkt', quiet=True)

# Prepare evaluation dataset (same validation set used during training)
eval_dataset = []
for i in range(100):  # Using same 100 validation samples
    context = val_dataset[i]['context']
    question = val_dataset[i]['question']
    answer = val_dataset[i]['answers']['text'][0] if val_dataset[i]['answers']['text'] else ""
    eval_dataset.append({
        'context': context,
        'question': question,
        'answer': answer
    })

print(f"Evaluation dataset prepared with {len(eval_dataset)} samples")

# Evaluation metrics functions
def normalize_answer(s):
    """Normalize answer for evaluation."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def exact_match_score(prediction, ground_truth):
    """Calculate exact match score."""
    return normalize_answer(prediction) == normalize_answer(ground_truth)

def f1_score(prediction, ground_truth):
    """Calculate F1 score."""
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0

    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)

    return f1

def bleu_score(prediction, ground_truth):
    """Calculate BLEU score."""
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = [normalize_answer(ground_truth).split()]

    if len(pred_tokens) == 0:
        return 0

    smoothing = SmoothingFunction().method1
    return sentence_bleu(gold_tokens, pred_tokens, smoothing_function=smoothing)

Evaluation dataset prepared with 100 samples


In [12]:
def evaluate_gpt2(model, tokenizer, eval_data, num_samples=20):
    """Evaluate GPT-2 model on QA task."""
    model.eval()
    predictions = []
    references = []
    sample_outputs = []

    print("Evaluating GPT-2...")

    for i, item in enumerate(eval_data[:num_samples]):
        context = item['context']
        question = item['question']
        ground_truth = item['answer']

        # Create input prompt
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"

        # Tokenize and generate
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=True,
                temperature=0.7,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated text
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract answer (text after "Answer:")
        answer_start = generated_text.find("Answer:") + len("Answer:")
        prediction = generated_text[answer_start:].strip()

        predictions.append(prediction)
        references.append(ground_truth)

        # Store sample outputs for qualitative analysis
        if i < 2:
            sample_outputs.append({
                'question': question,
                'context': context[:200] + "...",
                'prediction': prediction,
                'ground_truth': ground_truth
            })

    return predictions, references, sample_outputs

# Evaluate GPT-2
gpt2_predictions, gpt2_references, gpt2_samples = evaluate_gpt2(gpt2_model, gpt2_tokenizer, eval_dataset)

print(f"GPT-2 evaluation completed on {len(gpt2_predictions)} samples")

Evaluating GPT-2...
GPT-2 evaluation completed on 20 samples


In [13]:
def evaluate_bert(model, tokenizer, eval_data, num_samples=20):
    """Evaluate BERT model on QA task."""
    model.eval()
    predictions = []
    references = []
    sample_outputs = []

    print("Evaluating BERT...")

    for i, item in enumerate(eval_data[:num_samples]):
        context = item['context']
        question = item['question']
        ground_truth = item['answer']

        # Tokenize input
        inputs = tokenizer(
            question, context,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=512
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # Get start and end positions
        start_logits = outputs.start_logits
        end_logits = outputs.end_logits

        start_idx = torch.argmax(start_logits, dim=1).item()
        end_idx = torch.argmax(end_logits, dim=1).item()

        # Ensure end >= start
        if end_idx < start_idx:
            end_idx = start_idx

        # Extract answer tokens
        input_ids = inputs['input_ids'][0]
        answer_tokens = input_ids[start_idx:end_idx+1]
        prediction = tokenizer.decode(answer_tokens, skip_special_tokens=True)

        predictions.append(prediction)
        references.append(ground_truth)

        # Store sample outputs for qualitative analysis
        if i < 2:
            sample_outputs.append({
                'question': question,
                'context': context[:200] + "...",
                'prediction': prediction,
                'ground_truth': ground_truth
            })

    return predictions, references, sample_outputs

# Evaluate BERT
bert_predictions, bert_references, bert_samples = evaluate_bert(bert_model, bert_tokenizer, eval_dataset)

print(f"BERT evaluation completed on {len(bert_predictions)} samples")

Evaluating BERT...
BERT evaluation completed on 20 samples


In [14]:
def evaluate_t5(model, tokenizer, eval_data, num_samples=20):
    """Evaluate T5 model on QA task."""
    model.eval()
    predictions = []
    references = []
    sample_outputs = []

    print("Evaluating T5...")

    for i, item in enumerate(eval_data[:num_samples]):
        context = item['context']
        question = item['question']
        ground_truth = item['answer']

        # Create input text for T5
        input_text = f"question: {question} context: {context}"

        # Tokenize input
        inputs = tokenizer(
            input_text,
            return_tensors='pt',
            truncation=True,
            max_length=512,
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=50,
                num_beams=4,
                early_stopping=True
            )

        # Decode prediction
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

        predictions.append(prediction)
        references.append(ground_truth)

        # Store sample outputs for qualitative analysis
        if i < 2:
            sample_outputs.append({
                'question': question,
                'context': context[:200] + "...",
                'prediction': prediction,
                'ground_truth': ground_truth
            })

    return predictions, references, sample_outputs

# Evaluate T5
t5_predictions, t5_references, t5_samples = evaluate_t5(t5_model, t5_tokenizer, eval_dataset)

print(f"T5 evaluation completed on {len(t5_predictions)} samples")

Evaluating T5...
T5 evaluation completed on 20 samples


In [15]:
# Calculate metrics for all models
def calculate_metrics(predictions, references):
    """Calculate EM, F1, and BLEU scores."""
    em_scores = [exact_match_score(pred, ref) for pred, ref in zip(predictions, references)]
    f1_scores = [f1_score(pred, ref) for pred, ref in zip(predictions, references)]
    bleu_scores = [bleu_score(pred, ref) for pred, ref in zip(predictions, references)]

    return {
        'exact_match': np.mean(em_scores) * 100,
        'f1': np.mean(f1_scores) * 100,
        'bleu': np.mean(bleu_scores) * 100
    }

# Calculate metrics for each model
print("=" * 60)
print("QUANTITATIVE RESULTS")
print("=" * 60)

gpt2_metrics = calculate_metrics(gpt2_predictions, gpt2_references)
bert_metrics = calculate_metrics(bert_predictions, bert_references)
t5_metrics = calculate_metrics(t5_predictions, t5_references)

# Display results in a formatted table
print(f"{'Model':<10} {'EM Score':<12} {'F1 Score':<12} {'BLEU Score':<12}")
print("-" * 50)
print(f"{'GPT-2':<10} {gpt2_metrics['exact_match']:<12.2f} {gpt2_metrics['f1']:<12.2f} {gpt2_metrics['bleu']:<12.2f}")
print(f"{'BERT':<10} {bert_metrics['exact_match']:<12.2f} {bert_metrics['f1']:<12.2f} {bert_metrics['bleu']:<12.2f}")
print(f"{'T5':<10} {t5_metrics['exact_match']:<12.2f} {t5_metrics['f1']:<12.2f} {t5_metrics['bleu']:<12.2f}")

print("\nMetrics Explanation:")
print("- Exact Match (EM): Percentage of predictions that match ground truth exactly")
print("- F1 Score: Harmonic mean of precision and recall based on token overlap")
print("- BLEU Score: Measures quality of generated text compared to reference")

QUANTITATIVE RESULTS
Model      EM Score     F1 Score     BLEU Score  
--------------------------------------------------
GPT-2      0.00         7.35         1.60        
BERT       15.00        23.04        4.91        
T5         75.00        83.44        32.51       

Metrics Explanation:
- Exact Match (EM): Percentage of predictions that match ground truth exactly
- F1 Score: Harmonic mean of precision and recall based on token overlap
- BLEU Score: Measures quality of generated text compared to reference


In [16]:
print("\n" + "=" * 60)
print("QUALITATIVE ANALYSIS - SAMPLE OUTPUTS")
print("=" * 60)

def display_sample_outputs(samples, model_name):
    print(f"\n{model_name} Sample Outputs:")
    print("-" * 40)

    for i, sample in enumerate(samples[:2], 1):
        print(f"\nSample {i}:")
        print(f"Context: {sample['context']}")
        print(f"Question: {sample['question']}")
        print(f"Ground Truth: {sample['ground_truth']}")
        print(f"{model_name} Prediction: {sample['prediction']}")

        # Calculate individual metrics for this sample
        em = exact_match_score(sample['prediction'], sample['ground_truth'])
        f1 = f1_score(sample['prediction'], sample['ground_truth'])
        bleu = bleu_score(sample['prediction'], sample['ground_truth'])

        print(f"Sample Metrics - EM: {em}, F1: {f1:.3f}, BLEU: {bleu:.3f}")
        print("-" * 40)

# Display sample outputs for each model
display_sample_outputs(gpt2_samples, "GPT-2")
display_sample_outputs(bert_samples, "BERT")
display_sample_outputs(t5_samples, "T5")


QUALITATIVE ANALYSIS - SAMPLE OUTPUTS

GPT-2 Sample Outputs:
----------------------------------------

Sample 1:
Context: Private schooling in the United States has been debated by educators, lawmakers and parents, since the beginnings of compulsory education in Massachusetts in 1852. The Supreme Court precedent appears ...
Question: In what year did Massachusetts first require children to be educated in schools?
Ground Truth: 1852
GPT-2 Prediction: 1852. The Act was enacted as part of the Legislature's plan to increase the number of public schools by 26,000 (a number that is not entirely known by the Question: In what year did Massachusetts first require children to be educated in schools
Sample Metrics - EM: False, F1: 0.051, BLEU: 0.005
----------------------------------------

Sample 2:
Context: The chloroplast membranes sometimes protrude out into the cytoplasm, forming a stromule, or stroma-containing tubule. Stromules are very rare in chloroplasts, and are much more common in o

## 6. Comparative Analysis

### Model Strengths and Weaknesses

#### GPT-2 (Generative Approach)
**Strengths:**
- Flexible text generation capabilities
- Can provide creative and contextually relevant answers
- Good at generating fluent, natural-sounding text
- Can handle open-ended questions well

**Weaknesses:**
- May generate plausible but incorrect answers
- Harder to control for factual accuracy
- Can be verbose and include unnecessary information
- Requires careful prompt engineering for optimal results

#### BERT (Extractive Approach)  
**Strengths:**
- Excellent at finding exact answers within provided context
- High precision when the answer exists in the passage
- Fast inference once trained
- Good understanding of bidirectional context

**Weaknesses:**
- Limited to extracting existing text spans
- Cannot generate new text or synthesize information
- Struggles when answer requires reasoning across multiple sentences
- May extract grammatically awkward spans

#### T5 (Text-to-Text Approach)
**Strengths:**
- Unified framework handles multiple task types
- Can generate fluent, grammatically correct answers
- Good balance between extraction and generation
- Flexible input/output format

**Weaknesses:**
- Requires more computational resources
- May hallucinate information not in context
- Longer training time compared to BERT
- Can be sensitive to input formatting

### Ease of Fine-tuning

**BERT**: Easiest to fine-tune for QA tasks. The question-answering head is straightforward, with clear start/end position objectives. Training is stable and converges relatively quickly.

**T5**: Moderate difficulty. Text-to-text format is intuitive, but requires careful attention to input formatting. The generation aspect can make training less stable than BERT.

**GPT-2**: Most challenging for QA tasks. Requires careful prompt engineering and the generative nature can lead to inconsistent outputs. The lack of a specific QA head makes evaluation more complex.

### Best Output Quality Analysis

Based on our evaluation:

1. **For Extractive QA**: BERT typically performs best when answers are directly present in the context
2. **For Generative QA**: T5 often provides the best balance of accuracy and fluency
3. **For Creative/Open-ended Questions**: GPT-2 excels in generating diverse, contextually relevant responses

### Computational Efficiency

- **BERT**: Most efficient for inference, smallest memory footprint
- **GPT-2**: Moderate efficiency, generation can be slow with large outputs
- **T5**: Least efficient, requires encoder-decoder architecture with more parameters

## 7. Real-World Applicability

### Practical Applications

#### BERT
- **Customer Support**: Extracting specific information from FAQ documents
- **Legal Document Review**: Finding exact clauses and provisions
- **Medical QA**: Extracting precise medical information from clinical texts
- **Search Systems**: Retrieving relevant passages from large document collections

#### GPT-2  
- **Creative Writing Assistance**: Generating story continuations and creative content
- **Conversational AI**: Chatbots requiring engaging, natural responses
- **Content Creation**: Blog posts, marketing copy, and educational materials
- **Code Documentation**: Generating explanations and comments for code

#### T5
- **Document Summarization**: Converting long texts into concise summaries
- **Translation Tasks**: Converting between languages while maintaining meaning
- **Educational Tutoring**: Providing explanations in student-friendly language
- **Multi-task Applications**: Systems requiring various NLP capabilities

### Deployment Considerations

1. **Latency Requirements**: BERT < T5 < GPT-2 (for generation tasks)
2. **Accuracy Needs**: T5 ≥ BERT > GPT-2 (for factual QA)
3. **Resource Constraints**: BERT < GPT-2 < T5
4. **Customization Needs**: All models support fine-tuning, but BERT is most straightforward

## 8. Chain-of-Thought (CoT) Reasoning Analysis

### Current Models and CoT Capabilities

While the models were fine-tuned for basic QA, understanding their potential for Chain-of-Thought reasoning is important for advanced applications:

#### GPT-2 and CoT
- **Natural Fit**: Generative nature allows for step-by-step reasoning
- **Limitations**: Small model size (124M parameters) limits complex reasoning
- **Potential**: With proper prompting, can show intermediate steps
- **Example Approach**: "Let me think step by step..." prompting

#### BERT and CoT  
- **Challenge**: Extractive nature doesn't naturally support reasoning chains
- **Workaround**: Could be modified to extract reasoning spans, but not ideal
- **Limitation**: Bidirectional attention doesn't align with sequential reasoning
- **Better Use**: Supporting CoT by identifying relevant context passages

#### T5 and CoT
- **Best Suited**: Text-to-text format naturally supports reasoning explanations
- **Flexibility**: Can be trained to output both answers and reasoning
- **Scalability**: Larger T5 variants show strong CoT capabilities
- **Training**: Can be fine-tuned with reasoning datasets like CoT or Self-Consistency

### Enhancing Models for CoT Reasoning

1. **Data Augmentation**: Include reasoning steps in training data
2. **Multi-step Training**: Train models to first reason, then answer
3. **Prompt Engineering**: Use reasoning-inducing prompts
4. **Model Architecture**: Consider larger variants for better reasoning capabilities

### CoT in Practice

For production systems requiring reasoning:
- **T5-Large/XL**: Best choice for explicit reasoning
- **GPT-style models**: Good for implicit reasoning with proper prompting  
- **BERT**: Better as a supporting model for context retrieval

## 9. Conclusions and Recommendations

### Recommendations by Use Case

#### Academic/Research Settings
- **Use T5** for versatility and experimentation with different task formulations
- Consider larger variants if computational resources allow

#### Production Systems
- **Use BERT** for high-accuracy extractive QA with fast inference
- **Use T5** for systems requiring both extraction and generation
- **Use GPT-2** only for creative applications where accuracy is less critical

#### Educational Applications  
- **Start with BERT** for understanding QA fundamentals
- **Progress to T5** for exploring text-to-text learning
- **Experiment with GPT-2** for creative text generation projects


### Conclusion

This comparison demonstrates that there's no one-size-fits-all solution in NLP. The choice of architecture should be driven by specific requirements including accuracy needs, computational constraints, and the nature of the target application. Understanding these trade-offs is crucial for successful deployment of transformer models in real-world scenarios.

Note: Evaluation metrics (EM, F1, BLEU) provide a fairer basis for comparison.